# **Underpriced Cars**

### Active listings as at **2025.09.15**

- Seperate Testing Environment of the market_model
- Training & Plotting in the market_model Notebook



*In the best-case scenario, all active listings on the site would be predicted here. However, I don't have enough inactive listings for training yet, so I have to include some active listings in the training and validation aswell. This means those listings can't be tested / predicted here afterward, obviously.*


---

In [2]:
import torch
import pandas as pd
from torch.utils.data import DataLoader

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import sys

from modeling.dataset import CarPriceDataset
from modeling.model import MLPCarPriceRegressionNet_V1, MLPCarPriceRegressionNet_V2
from modeling.test import test_model
from modeling.plots_and_metrics import find_undervalued_cars

from preprocessing.preprocess import PreProcessor

## **Testing Prep & Testing**

---

In [5]:
MODEL_NAME = 'market_model'
CAR_DETAILS_DATASET_TEST = 'car_details_174722_20250916.csv'
CAR_DATA_DATASET = 'car_data_174722_20250916.csv'

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
car_data = pd.read_csv(f'data/raw/{CAR_DATA_DATASET}')
df_test = pd.read_csv(f'workspace/test_datasets/{CAR_DETAILS_DATASET_TEST}')

preprocessor = PreProcessor()
preprocessor.load(f'workspace/config/{MODEL_NAME}.pkl')

batch_size = 64
X_num_test, X_cat_test, y_test = preprocessor.transform(df_test, is_training=False, include_target=True)
test_dataset = CarPriceDataset(X_num_test, X_cat_test, y_test)
dataloader_test = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

model = MLPCarPriceRegressionNet_V2(f'workspace/config/{MODEL_NAME}.pkl')
model.load_state_dict(torch.load(f'workspace/models/{MODEL_NAME}.pt'))

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

results = test_model(model, dataloader_test)

## **Potential good deals**

Cars my model thinks are undervalued by more than 50% +

***Note: Some apparent "bargains" may be false positives due to misleading price listings.***

Common issues include:

- Prices displayed are only the down payments
- There may still be loans on the car
- No picture is included in the listing (Not neccessarily error, but it probably means its damaged inside) 

These are almost always specified in the description, so cars should always be verified through the links

In [6]:
display(find_undervalued_cars(results, df_test, car_data, undervalued_threshold=50))

actual_price,predicted_price,undervalued_percent,manufacturer,model,year,kilometers,kw,url,first_seen
"11,490,009 Ft","29,104,664 Ft",+153.3%,MERCEDES-BENZ,GLE-OSZTÁLY,2018,"137,200 km",190,Car page,2025-08-11
"3,050,000 Ft","7,206,877 Ft",+136.3%,MERCEDES-BENZ,E-OSZTÁLY E 350,2012,"307,000 km",265,Car page,2025-08-11
"2,000,000 Ft","4,439,638 Ft",+122.0%,FORD,MUSTANG,2011,"243,895 km",224,Car page,2025-09-06
"2,200,000 Ft","4,802,793 Ft",+118.3%,BMW,X SOROZAT X3,2012,"290,000 km",225,Car page,2025-09-05
"2,090,000 Ft","4,364,602 Ft",+108.8%,VOLKSWAGEN,TOURAN,2016,"200,000 km",110,Car page,2025-08-03
"6,000,000 Ft","11,959,803 Ft",+99.3%,PEUGEOT,208,2022,"43,300 km",100,Car page,2025-09-06
"7,999,997 Ft","15,663,817 Ft",+95.8%,JEEP,CHEROKEE GRAND CHEROKEE,2014,"58,000 km",344,Car page,2025-08-05
"3,490,000 Ft","6,738,350 Ft",+93.1%,INFINITI,QX QX70,2017,"210,000 km",175,Car page,2025-09-06
"4,000,000 Ft","7,591,390 Ft",+89.8%,KIA,CEE'D,2023,"81,400 km",118,Car page,2025-09-15
"3,899,998 Ft","7,356,116 Ft",+88.6%,TOYOTA,MR 2,2020,"225,000 km",103,Car page,2025-08-23
